# Ordering: does recovered demand make a better order?

The last stage, and the one that actually answers the business question. Everything upstream produced a forecast; this turns that forecast into an order quantity and checks what that order would have cost.

Three questions, one table each:

1. **Are the forecast's bands trustworthy?** Calibration.
2. **Does recovery make a better order?** The headline result.
3. **Does that hold no matter what a stockout actually costs?** The cost sweep.

Sections 1-3 are **validation only**. Section 4 opens the sealed test week, a deliberate, declared action, off by default. The full write-up, with every claim marked by which window it rests on, lives in notebook 05.

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, "..")   # notebooks/ is one level down, so the repo root goes on the path

import warnings
warnings.filterwarnings('ignore')

import json

import pandas as pd

from src import conformal, forecast, orders
from src.utils import config, plots

# The four model/target combinations this whole notebook calibrates and orders.
ARMS = [(family, target) for family in ("tft", "xgb") for target in ("recovered", "raw")]
# The two prediction-interval widths reported below (columns: nominal, lo quantile, hi quantile, file tag).
BANDS = [(0.80, "q10", "q90", "wide80"), (0.95, "q025", "q975", "wide95")]
PERIOD = "validation"

## 1. Are the bands trustworthy?

An 80% band should contain the true value 80% of the time. Raw model output doesn't. It's over-confident, as most models' raw prediction intervals are. Split-conformal (CQR) fits one widening offset on the calibration window and applies it, without retraining anything.

In [ ]:
RUN_CONFORMAL = True   # ~90s. The corrected parquets are intermediates and not in git,
                       # so this rebuilds them; set False to reuse a local run.

# Built here (cheap - it's one parquet read) rather than only inside the RUN_CONFORMAL branch,
# so `master` is always available for section 4 below even if this flag gets set to False.
master = forecast.build_master_frame()

if RUN_CONFORMAL:
    # Calibrate every arm, at both band widths, on the validation window only.
    for family, target in ARMS:
        for nominal, lo, hi, tag in BANDS:
            conformal.run(nominal, lo, hi, tag=tag, forecast_tag=target, family=family,
                          eval_periods=(PERIOD,), master=master, verbose=False)

# Read back the coverage numbers conformal.run just wrote for one arm (tft_recovered) - the
# before/after correction story is the same shape for every arm, so one is enough to show here.
rows = []
for nominal, _, _, tag in BANDS:
    period = json.loads(config.conformal_results(
        tag, family="tft", target="recovered").read_text())["periods"][PERIOD]
    ci = period["coverage_ci"]
    rows.append({"band": f"{nominal:.0%}",
                 "should cover": f"{nominal:.0%}",
                 "covers before": f"{period['uncorrected']['coverage']:.0%}",
                 "covers after": f"{period['corrected']['coverage']:.0%}",
                 # day-block bootstrap, not Kupiec - see the note below
                 "95% CI": f"[{ci['ci_low']:.1%}, {ci['ci_high']:.1%}]"})
print(pd.DataFrame(rows).to_string(index=False))

**Result: calibration works, and it's honest about what's still left over.**

The bands started too narrow: an 80% band covering only 74%. After correction they cover 83% and 97%, so the remaining error is 2-3 percentage points, and it's on the **safe** side (slightly too wide, rather than still overconfident).

Two things worth stating plainly here, because both look like failures at first glance and neither actually is:

- **The classical test (Kupiec) rejects this, and that's fine to ignore.** It assumes every row is an independent observation. These rows are 5,601 products all sharing the same date, so one busy Saturday moves thousands of them together at once. Measured, the day-to-day spread is **13.7×** what independence would predict. The effective sample size is really about 3,500, not 48,000, so the test is being handed fourteen times more evidence than actually exists, and it rejects a 2-point miss that doesn't matter. What we report instead is coverage with a confidence interval built by resampling whole days.
- **A better conformal method was looked for, and it doesn't exist here.** Multiplicative band-scaling and per-censoring-band (Mondrian) offsets were both tried against plain CQR on a like-for-like harness that never touches the test week. All three land **within 0.0003 of each other**. So the remaining gap isn't about the formula, it's distance in time: coverage decays by about 2.5 points per fortnight of extrapolation. `conformal.FORWARD_DRIFT_INFLATION` is the measured fix for that (ask for 83% coverage to land near 80% a window later), and it only kicks in when forecasting forward, which in this project means the test week.

## 2. Does recovery make a better order?

This comparison only makes sense if we hold **demand met** fixed. At a fixed cost ratio each arm picks its own order quantity, so an arm that simply orders more will stock out less and waste more, and a table built that way can't tell "forecasts better" apart from "just orders harder".

So instead: every arm is held to meeting **95% of demand**, and the real question becomes what each one has to waste to get there. Scored on full-shelf days, where recorded sales are the true demand, which happens to be the regime that *penalises* recovery, since those are the quiet days where it tends to over-order.

In [ ]:
# Load all four arms' forecasts, then find the order quantile each one needs to hit 95% demand met.
frames = {f"{family}_{target}": orders.load_forecast(period=PERIOD, family=family, target=target)
          for family, target in ARMS}

demand_met = orders.at_demand_met(frames, target_demand_met=0.95, verbose=False)

# Pivot to the shape of the actual claim: raw against recovered, one row per model family.
s = demand_met.set_index("arm")
comparison = pd.DataFrame([
    {"model": name,
     "waste on raw sales": f"{s.loc[fam + '_raw', 'waste_pct']:.1f}%",
     "waste on recovered": f"{s.loc[fam + '_recovered', 'waste_pct']:.1f}%",
     "waste saved": f"{s.loc[fam + '_recovered', 'waste_pct'] - s.loc[fam + '_raw', 'waste_pct']:+.1f} pts",
     "raw orders at": f"q{s.loc[fam + '_raw', 'order_percentile']:.2f}",
     "recovered orders at": f"q{s.loc[fam + '_recovered', 'order_percentile']:.2f}"}
    for fam, name in [("tft", "TFT"), ("xgb", "XGBoost")]])
print("every arm held to 95% of demand met, on full-shelf days")
print(comparison.to_string(index=False))

**Result: recovery cuts waste at the same availability, in both models.**

Same shelf availability, less food ending up in the bin: 2.6 points less for the TFT, 5.3 for XGBoost. Two unrelated model families, same direction.

The `orders at` columns are the actual mechanism, and worth explaining. To meet 95% of demand, the **raw** models have to order at roughly the **79th percentile** of their own forecast, while the **recovered** models only need the **68th-70th**. A model trained on censored sales runs systematically low, so it has to be pushed harder to keep shelves full, and pushing it harder is exactly what fills the bins. Recovery removes the need for that compensation, and the waste saving is just that compensation no longer being paid.

TFT vs. XGBoost is a much smaller gap than recovery vs. raw. The two architectures are close on accuracy (TFT ahead by about 4%), which tells us it's the recovery layer doing the work here, not the choice of model.

## 3. Does it hold no matter what a stockout actually costs?

Real costs aren't in the data, so the answer shouldn't depend on any one guess. `c_u/c_o` is how much worse an empty shelf is than a binned unit, swept from 2× to 9×.

In [ ]:
# Run the full newsvendor + nine-ratio cost sweep for the recovered TFT arm.
head = orders.run(*frames["tft_recovered"], period=PERIOD, regime="observed", verbose=False)
sweep = head["sweep"]

# Just three representative ratios (2x, 4x, 9x) from the full sweep, for a readable summary.
view = sweep[sweep["c_u"].isin([2.0, 4.0, 9.0])]
print("ordering from the recovered TFT, across the plausible range of stockout cost")
print(pd.DataFrame({
    "a stockout costs": view["c_u"].map(lambda v: f"{v:.0f}x a bin"),
    "so we order at": view["q_star"].map(lambda v: f"q{v:.2f}"),
    "TFT stockouts": view["stockout_pct_model"].map(lambda v: f"{v:.1f}%"),
    "naive stockouts": view["stockout_pct_naive"].map(lambda v: f"{v:.1f}%"),
    "TFT cost vs naive": view["cost_vs_naive_pct"].map(lambda v: f"-{v:.0f}%"),
}).to_string(index=False))

# The full nine-ratio sweep, drawn as a line against the naive rule.
plots.plot_cost_sweep(sweep, save_path=config.PLOTS_DIR / "cost_sweep.png")

**Result: cheaper than the status quo at every ratio tested, 20.5% to 72.6% lower cost.**

The naive rule ("order what sold this day last week") stocks out **41.7%** of the time. Ours ranges from 77% down to 2%, depending on how much a stockout is assumed to cost. The conclusion doesn't depend on picking any one cost ratio, that's the whole point of sweeping it.

**Read the waste column next to the cost column, never on its own.** Waste against the naive rule turns negative from `c_u = 2` upward, but the naive rule only wastes so little *because it's empty two days in five*. Any policy that actually keeps shelves stocked is going to lose on waste while winning on cost, so cost is the only column that fairly prices both kinds of mistake, on the terms the ratio declares.

The defensible claim is the one from §2: **at equal availability, recovery wastes less.** That's a genuinely like-for-like comparison, and it holds in both model families.

## 4. The test week

Off by design. The forecast, calibration and ordering stages all default to validation, and reaching the test week takes deliberately naming it.

**It's already been opened once, before the ordering code was corrected.** The calibration numbers from that earlier run are legitimate and kept as-is; the ordering numbers came from a version whose stockout rate was an identity rather than a real measurement, and those are not kept. Turning this on now is a **declared second look**, and the write-up says so.

**All four arms.** The two TFT test forecasts are produced right here from their saved checkpoints, no retraining, about 25s each, and since the model early-stopped on validation alone, nothing about the test week gets to inform it. The two XGBoost test forecasts come from **notebook 03**, whose fit cell passes `periods=(..., "test")`. Whatever's found on disk gets calibrated and ordered; anything missing is skipped **and named explicitly**, so no table downstream can quietly end up describing fewer arms than it looks like it does.

**One wrinkle, handled rather than ignored.** `forecast.load` needs the *encoder length* each checkpoint was fit with, and that value isn't stored on the checkpoint itself, it's read from that tag's tuning table. Only the recovered arm has one: `tft_best_raw.ckpt` exists, but its search was never saved. So the raw arm's encoder length is **borrowed** from the recovered arm, which is a reasonable thing to do since the raw-vs-recovered comparison only ever changes the target, never the configuration.

That justification isn't just taken on faith, though. A wrong encoder length wouldn't raise an error, it would silently build a differently-conditioned dataset and hand back a forecast that still looks plausible. So the borrowed value gets **verified**: the raw model is rebuilt at that length, re-forecast over the **validation** window, and diffed against the raw validation forecast already saved on disk. The cell asserts that gap is below a calibrated tolerance before it'll produce a test forecast at all. The borrowed value lands at 7.6e-06 (basically float32 noise), while encoder lengths of 2, 4 and 7 land at 0.58, 0.43 and 2.9. So the check discriminates by five orders of magnitude, not by a hopeful threshold, and it never reads anything from the sealed data to do it.

The forward-drift correction switches itself on here and nowhere else: the test week is the only window that sits after the calibration set, and `conformal._is_forward` reads that straight off the frozen calendar rather than being told to.

In [ ]:
OPEN_TEST_WEEK = True   # a deliberate, declared, one-time action - see the note above

if OPEN_TEST_WEEK:
    # Calibrate and order every arm that now has a test forecast. XGBoost's come from notebook 03,
    # whose fit cell passes periods=(..., "test"). Anything missing is SKIPPED AND NAMED, so no table
    # downstream can quietly describe fewer arms than it appears to. eval_periods carries validation
    # too, so one JSON per arm holds both windows and the conclusion can compare near against far.
    test_arms = []
    for family, target in ARMS:
        if not config.forecast_parquet("test", target, family).exists():
            print(f"skipped {family}_{target} - no test forecast on disk")
            continue
        test_arms.append((family, target))
        for nominal, lo, hi, tag in BANDS:
            conformal.run(nominal, lo, hi, tag=tag, forecast_tag=target, family=family,
                          eval_periods=("validation", "test"), master=master, verbose=False)

    test_frames = {f"{f}_{t}": orders.load_forecast(period="test", family=f, target=t)
                   for f, t in test_arms}
    print(f"\ncalibrated and ordered on the test week: {', '.join(test_frames)}\n")
    orders.at_demand_met(test_frames, target_demand_met=0.95)
else:
    test_frames = {}
    print("OPEN_TEST_WEEK=False - nothing in this notebook has read the test week")

---

Everything that reads from here on (the recovery/accuracy/calibration battery, the demand-met comparison across both windows, the search for a good operating point, and the project's full write-up) is assembled in **notebook 05**, computed fresh from the artifacts this notebook (and notebooks 01-03) wrote, rather than copied over by hand.